# Day 2, Session 3 — Amazon SageMaker Part 2: Deployment & MLOps

Continue in **the same SageMaker Studio kernel** as Session 2 if you still have it open
— the `est` object from that notebook already holds your trained model, so you can skip
straight to Cell 4.

Starting fresh, or in a new kernel? Run Cell 0 first to reconnect to your completed
training job before continuing.

In [ ]:
# Cell 0 — only needed if you are starting a NEW kernel/session
# Reattaches to the most recently completed SKLearn training job so `est` exists again.
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "ai-lab-<your-username>"

# Replace with the exact Training Job name shown in the SageMaker console
# (SageMaker → Training → Training jobs → copy the Name column)
training_job_name = "<your-training-job-name>"
est = SKLearn.attach(training_job_name)

In [ ]:
# Cell 4 — deploy the trained model to a real-time endpoint (takes ~5 min)
predictor = est.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)

In [ ]:
# Cell 5 — invoke it: the model answers over the network
import pandas as pd

recruits = pd.DataFrame(
    [[24, 48, 23], [40, 15, 36]],
    columns=["age", "pushups_per_min", "run_5km_minutes"],
)
print(predictor.predict(recruits))

**What you should see:** Cell 4 prints dashes for ~5 minutes while the endpoint machine
starts, then `----!`. Cell 5 returns `['high', 'low']` — your model, answering over the
internet. In the console: **SageMaker → Endpoints** shows status `InService`.

Open **CloudWatch** from the endpoint's Monitoring tab to see the invocation count and
latency graphs — this is MLOps monitoring in its simplest form.

**Discussion:** walk the MLOps loop — data → train → deploy → monitor → drift
detected → retrain → redeploy. Day 4 covers the train/fine-tune stages in depth.

In [ ]:
# Cell 6 — the most professional line you will write today
predictor.delete_endpoint()

**Verify in the console:** SageMaker → Endpoints must be **EMPTY** before you take a
break.

> **Cost rule:** a forgotten `ml.m5.large` endpoint costs roughly $80–100 per month.
> Training jobs shut themselves down; endpoints never do. The habit: deploy → test →
> **DELETE**. No exceptions, today or ever.